# 第 17 章 · 可微仿真与 SHAC：SuperMimic 之前，解析梯度是怎么来的

配套讲义：仓库里 `robot-motion-rl/notes/lessons/17-shac.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（刚度、β、高度是编的；H = 1000、h = 32 取自 SHAC 论文）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 硬接触 | k·max(0, −z) | `hard_contact_force`、`hard_contact_grad` | z = −0.005：5.00 N、−1000 |
| 第二步 平滑接触 | softplus | `soft_contact_force`、`soft_contact_grad` | z = 0：6.93 N、−500；z = 0.02：1.27 N、−119 |
| 第三步 窗口 | ⌈H/h⌉ | `windows(1000, 32)` | 32 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `hard_contact_force`**：硬接触（罚函数）：脚陷进地面（z < 0）才有力 f = k·(−z)，否则 0。

In [ ]:
def hard_contact_force(z, k=1000.0):
    """硬接触（罚函数）：脚陷进地面（z < 0）才有力 f = k·(−z)，否则 0。"""
    return k * max(0.0, -z)

**函数 `hard_contact_grad`**：硬接触对高度的导数：离地时 0，触地时 −k，在 z = 0 处跳变。

In [ ]:
def hard_contact_grad(z, k=1000.0):
    """硬接触对高度的导数：离地时 0，触地时 −k，在 z = 0 处跳变。"""
    return -k if z < 0 else 0.0

**函数 `soft_contact_force`**：平滑接触：把 max(0, −z) 换成 softplus，β 越小越接近硬接触。f = k·β·log(1 + exp(−z/β))。

In [ ]:
def soft_contact_force(z, k=1000.0, beta=0.01):
    """平滑接触：把 max(0, −z) 换成 softplus，β 越小越接近硬接触。f = k·β·log(1 + exp(−z/β))。"""
    return float(k * beta * np.log1p(np.exp(-z / beta)))

**函数 `soft_contact_grad`**：平滑接触的导数：−k·sigmoid(−z/β)，从 0 平滑过渡到 −k。

In [ ]:
def soft_contact_grad(z, k=1000.0, beta=0.01):
    """平滑接触的导数：−k·sigmoid(−z/β)，从 0 平滑过渡到 −k。"""
    return float(-k / (1 + np.exp(z / beta)))

**函数 `windows`**：SHAC：长度 H 的任务切成多少个长度 h 的短窗口（向上取整）。

In [ ]:
def windows(total_horizon, window):
    """SHAC：长度 H 的任务切成多少个长度 h 的短窗口（向上取整）。"""
    return -(-total_horizon // window)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 硬接触 vs 平滑接触")
for z in (0.02, 0.005, 0.0, -0.005):
    print(f"z = {z:+.3f} m：硬 f = {hard_contact_force(z):.2f}，df/dz = {hard_contact_grad(z):.0f}；"
          f"平滑 f = {soft_contact_force(z):.2f}，df/dz = {soft_contact_grad(z):.1f}")

In [ ]:
print("\n第三步 短窗口")
print("SHAC 论文：H = 1000 → h = 32 的窗口数 =", windows(1000, 32))

## 3. 代码里才看得到的

- 用 `np.log1p(x)` 而不是 `np.log(1 + x)`：x 很小时前者更精确；z 很大（脚很高）时 $e^{-z/\beta}$ 接近 0，`log1p` 不会丢精度。

## 4. 自己改一改

- 把 β 改成 0.001，看 z = 0.02 处的虚假推力降到多少、z = 0 处的导数怎么变。
- 取 β = 0.003，写一个小循环让脚从 z = 0.03 落到 −0.01，比较两种接触力随 z 的曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 收音机给了电路图，仿真器自己出导数；
> 脚在空中导数零，一碰地跳到负一千，策略学不会踩；
> softplus 把台阶磨成坡，导数连续，代价是隔空有力；
> SHAC 对真仿真求导要磨平，SuperMimic 对学的模型求导随便用。